# Tensor-RL Experiments
Demonstration notebook for all training scenarios. Most cells print the CLI command to run rather than executing a full sweep (training takes minutes–hours). The final section shows how to load saved results and run analysis.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

def run_name(env, algo, network, rank, tensorize='all', seed=42):
    tz = f'_tz{tensorize}' if tensorize != 'all' else ''
    return f'{env}_{algo}_{network}_rank{rank}{tz}_seed{seed}'

ENV = 'MiniGrid-Empty-5x5-v0'
print('Setup complete.')

## 1. Standard DQN Variants
Three algorithms, each using standard (non-tensorized) linear layers.

In [ ]:
algos = ['dqn', 'double_dqn', 'dueling_dqn']
for algo in algos:
    cmd = f'conda run -n tensor python train.py --env {ENV} --algo {algo} --network standard --episodes 500'
    print(cmd)

## 2. Tensor Decomposition Sweep
Grid over network types and ranks using Double DQN.

In [ ]:
networks = ['standard', 'cp', 'tucker', 'tt']
ranks = [2, 4, 8]
for net in networks:
    for rank in ranks:
        if net == 'standard' and rank != ranks[0]:
            continue  # standard ignores rank
        cmd = (f'conda run -n tensor python train.py '
               f'--algo double_dqn --network {net} --rank {rank} --episodes 500')
        print(cmd)

## 3. Selective Tensorization
Compare `--tensorize all` vs `--tensorize hidden` for CP and TT.
With `hidden`, only the hidden layers are decomposed; the final output layer stays `nn.Linear`.

In [ ]:
for net in ['cp', 'tt']:
    for tz in ['all', 'hidden']:
        cmd = (f'conda run -n tensor python train.py '
               f'--algo double_dqn --network {net} --rank 4 --tensorize {tz} --episodes 500')
        print(cmd)

## 4. Tabular Agents Demo
Run `TabularQAgent`, `SarsaAgent`, and `TensorizedTabularQAgent` directly on FrozenLake-v1 (discrete state space). This runs inline — no CLI needed.

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from agents.tabular_q import TabularQAgent, SarsaAgent, TensorizedTabularQAgent

def run_tabular(agent, env_id='FrozenLake-v1', n_steps=2000, seed=42):
    env = gym.make(env_id, is_slippery=False)
    state, _ = env.reset(seed=seed)
    rewards = []
    ep_reward = 0.0
    ep_rewards = []
    for _ in range(n_steps):
        action = agent.select_action(state)
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        agent.update(state, action, reward, next_state, done)
        ep_reward += reward
        if done:
            ep_rewards.append(ep_reward)
            ep_reward = 0.0
            state, _ = env.reset()
        else:
            state = next_state
    return ep_rewards

S, A = 16, 4  # FrozenLake 4x4
agents = {
    'TabularQ': TabularQAgent(S, A, epsilon_decay_steps=1000),
    'Sarsa': SarsaAgent(S, A, epsilon_decay_steps=1000),
    'TensorizedQ (rank=4)': TensorizedTabularQAgent(S, A, rank=4, epsilon_decay_steps=1000),
}

plt.figure(figsize=(9, 4))
for name, agent in agents.items():
    ep_rewards = run_tabular(agent, n_steps=2000)
    window = 10
    smoothed = np.convolve(ep_rewards, np.ones(window)/window, mode='valid') if len(ep_rewards) >= window else ep_rewards
    plt.plot(smoothed, label=name)
plt.xlabel('Episode'); plt.ylabel('Reward'); plt.title('Tabular Agents on FrozenLake-v1 (no slip)')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

## 5. Multi-Seed Run
Use `--seeds` to run the same config with multiple seeds and produce an aggregate JSON.

In [ ]:
cmd = ('conda run -n tensor python train.py '
       '--algo double_dqn --network cp --rank 4 --seeds 42,43,44 --episodes 500')
print(cmd)
# This produces:
#   sim/data/..._seed42.json
#   sim/data/..._seed43.json
#   sim/data/..._seed44.json
#   sim/data/..._agg.json   ← mean/std across seeds

## 6. Load and Analyse Results
After running training, use `analysis/stats.py` to inspect and compare runs.

In [ ]:
from analysis.stats import load_data, plot_learning_curves, plot_param_efficiency, plot_stability, run_statistical_tests

runs, agg_runs = load_data('sim/data')
print(f'Loaded {len(runs)} run(s) and {len(agg_runs)} aggregate file(s).')
for r in runs:
    print(f"  {r['name']}  —  {len(r['metrics'].get('rewards', []))} episodes")

In [ ]:
# Training reward curves (smoothed, with shaded std when multiple seeds)
plot_learning_curves(runs, agg_data=agg_runs, metric='rewards', smoothing=20)

In [ ]:
# Eval reward curves (greedy policy, no exploration)
plot_learning_curves(runs, agg_data=agg_runs, metric='eval_reward', smoothing=20)

In [ ]:
# Parameter efficiency: fewer params vs final performance
# Each point is one run; tensorized runs should be left of standard with comparable y-axis
plot_param_efficiency(runs)

In [ ]:
# Stability: gradient norm (left) and mean Q-value magnitude (right)
# High grad norm spikes = instability; Q divergence = overestimation
plot_stability(runs)

In [ ]:
# Statistical significance of performance differences between network types
# Uses Mann-Whitney U (non-parametric) on final-20-episode eval rewards
if len(runs) >= 2:
    results = run_statistical_tests(runs)
else:
    print('Need at least 2 runs for statistical comparison.')